In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project folder
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name.lower() in ["notebooks", "notebook"]:
    PROJECT_DIR = PROJECT_DIR.parent

# Find the dataset
possible_paths = [
    PROJECT_DIR / "data" / "raw" / "ChargingRecords.csv",
    PROJECT_DIR / "ChargingRecords.csv",
    Path.cwd() / "ChargingRecords.csv"
]

DATA_PATH = next(
    (path for path in possible_paths if path.exists()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "ChargingRecords.csv not found. "
        "Place it in the project folder or data/raw folder."
    )

raw_df = pd.read_csv(DATA_PATH)
df = raw_df.copy()

initial_rows = len(df)
initial_missing_cells = int(df.isnull().sum().sum())
initial_duplicate_rows = int(df.duplicated().sum())

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (72856, 13)


,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54,2022-09-15 23:59,185,20.36
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01,2022-09-14 21:31,90,10.19
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54,2022-09-14 19:54,60,6.78
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32,2022-09-30 0:16,344,37.65
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30,2022-09-26 0:30,300,33.81


In [9]:
print("Initial Data Quality Report")

print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nMissing Values by Column:")
display(
    df.isnull().sum()
    .to_frame("Missing Count")
)

print("\nData Types:")
display(
    df.dtypes.to_frame("Data Type")
)

Initial Data Quality Report
Total rows: 72856
Total columns: 13
Missing values: 0
Duplicate rows: 13

Missing Values by Column:


,Missing Count
UserID,0
ChargerID,0
ChargerCompany,0
Location,0
ChargerType,0
StartDay,0
StartTime,0
EndDay,0
EndTime,0
StartDatetime,0



Data Types:


,Data Type
UserID,int64
ChargerID,int64
ChargerCompany,int64
Location,str
ChargerType,int64
StartDay,str
StartTime,str
EndDay,str
EndTime,str
StartDatetime,str


In [10]:
# Remove spaces from column names
df.columns = df.columns.str.strip()

# Check required columns
required_columns = [
    "UserID",
    "ChargerID",
    "ChargerCompany",
    "Location",
    "ChargerType",
    "StartDay",
    "StartTime",
    "EndDay",
    "EndTime",
    "StartDatetime",
    "EndDatetime",
    "Duration",
    "Demand"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

# Remove extra spaces from text columns
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype("string").str.strip()

# Standardize location names
df["Location"] = (
    df["Location"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Convert numeric columns
numeric_columns = [
    "UserID",
    "ChargerID",
    "ChargerCompany",
    "ChargerType",
    "Duration",
    "Demand"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# Reconstruct precise timestamps using the date and time columns
original_start = pd.to_datetime(
    df["StartDatetime"], errors="coerce"
)

original_end = pd.to_datetime(
    df["EndDatetime"], errors="coerce"
)

start_from_parts = pd.to_datetime(
    df["StartDay"].astype(str) + " "
    + df["StartTime"].astype(str),
    errors="coerce"
)

end_from_parts = pd.to_datetime(
    df["EndDay"].astype(str) + " "
    + df["EndTime"].astype(str),
    errors="coerce"
)

# Use the separate date/time fields when available
df["StartDatetime"] = start_from_parts.fillna(original_start)
df["EndDatetime"] = end_from_parts.fillna(original_end)

# Replace infinite numeric values with missing values
df[numeric_columns] = df[numeric_columns].replace(
    [np.inf, -np.inf],
    np.nan
)

print("Column names and data types standardized.")
display(df.head())

C:\Users\kavee\AppData\Local\Temp\ipykernel_2392\1198024356.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=["object"]).columns:


Column names and data types standardized.


,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54:02,2022-09-15 23:59:13,185,20.36
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01:05,2022-09-14 21:31:04,90,10.19
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54:30,2022-09-14 19:54:29,60,6.78
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32:51,2022-09-30 00:16:42,344,37.65
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30:15,2022-09-26 00:30:14,300,33.81


In [11]:
# 1. Remove exact duplicate rows
rows_before = len(df)

df = df.drop_duplicates().copy()

duplicates_removed = rows_before - len(df)

print("Duplicate rows removed:", duplicates_removed)

# 2. Remove rows missing essential information
essential_columns = [
    "UserID",
    "ChargerID",
    "ChargerCompany",
    "Location",
    "ChargerType",
    "StartDatetime",
    "EndDatetime",
    "Demand"
]

rows_before = len(df)

df = df.dropna(
    subset=essential_columns
).copy()

missing_rows_removed = rows_before - len(df)

print("Rows removed due to missing essential values:",
      missing_rows_removed)

# 3. Remove negative demand values
negative_demand_mask = df["Demand"] < 0

negative_demand_removed = int(
    negative_demand_mask.sum()
)

df = df.loc[~negative_demand_mask].copy()

print("Negative demand rows removed:",
      negative_demand_removed)

# 4. Calculate elapsed time using timestamps
elapsed_minutes = (
    df["EndDatetime"] - df["StartDatetime"]
).dt.total_seconds() / 60

# Remove sessions with negative duration or end time before start
invalid_session_mask = (
    (elapsed_minutes < 0)
    | (df["Duration"] < 0)
)

invalid_sessions_removed = int(
    invalid_session_mask.sum()
)

df = df.loc[~invalid_session_mask].copy()

print("Invalid sessions removed:", invalid_sessions_removed)

Duplicate rows removed: 13
Rows removed due to missing essential values: 0
Negative demand rows removed: 0
Invalid sessions removed: 17


In [12]:
# Recalculate elapsed time after removing invalid sessions
elapsed_minutes = (
    df["EndDatetime"] - df["StartDatetime"]
).dt.total_seconds() / 60

# Identify missing or non-positive durations
# that can be repaired from timestamps
repair_mask = (
    df["Duration"].isna()
    | (df["Duration"] <= 0)
) & (
    (elapsed_minutes > 0)
    & (elapsed_minutes <= 1440)
)

duration_values_repaired = int(repair_mask.sum())

# Duration is stored in whole minutes
df.loc[repair_mask, "Duration"] = np.rint(
    elapsed_minutes.loc[repair_mask]
)

print("Duration values repaired:", duration_values_repaired)

# Identify exact zero-length sessions with non-positive duration
elapsed_minutes = (
    df["EndDatetime"] - df["StartDatetime"]
).dt.total_seconds() / 60

zero_length_mask = (
    (elapsed_minutes == 0)
    & (df["Duration"].fillna(0) <= 0)
)

zero_length_rows_removed = int(
    zero_length_mask.sum()
)

# Remove durations that remain missing or invalid
# and cannot be repaired reliably
unresolved_duration_mask = (
    df["Duration"].isna()
    | (
        (df["Duration"] <= 0)
        & ~(
            (elapsed_minutes > 0)
            & (elapsed_minutes <= 1440)
        )
    )
)

other_unresolved_rows_removed = int(
    (unresolved_duration_mask & ~zero_length_mask).sum()
)

df = df.loc[~unresolved_duration_mask].copy()

# Restore integer data types for ID and duration columns
integer_columns = [
    "UserID",
    "ChargerID",
    "ChargerCompany",
    "ChargerType",
    "Duration"
]

for col in integer_columns:
    df[col] = df[col].round().astype("int64")

print("Zero-length rows removed:", zero_length_rows_removed)
print("Other unresolved duration rows removed:",
      other_unresolved_rows_removed)

print("\nCleaning completed.")
print("Remaining rows:", len(df))

Duration values repaired: 691
Zero-length rows removed: 27
Other unresolved duration rows removed: 0

Cleaning completed.
Remaining rows: 72799


In [13]:
# Compare stored duration with timestamp-based elapsed time
elapsed_minutes = (
    df["EndDatetime"] - df["StartDatetime"]
).dt.total_seconds() / 60

duration_difference = (
    df["Duration"] - elapsed_minutes
).abs()

# Flag differences greater than 2 minutes
mismatch_mask = duration_difference > 2

print("Duration/timestamp mismatches greater than 2 minutes:",
      int(mismatch_mask.sum()))

# Save flagged records for manual review
anomaly_columns = [
    "UserID",
    "ChargerID",
    "Location",
    "StartDatetime",
    "EndDatetime",
    "Duration",
    "Demand"
]

duration_anomalies = df.loc[
    mismatch_mask,
    anomaly_columns
].copy()

duration_anomalies["ElapsedDurationMinutes"] = (
    elapsed_minutes.loc[mismatch_mask].round(2)
)

duration_anomalies["DurationDifferenceMinutes"] = (
    duration_difference.loc[mismatch_mask].round(2)
)

# Detect duration outliers using the IQR method
q1 = df["Duration"].quantile(0.25)
q3 = df["Duration"].quantile(0.75)

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outlier_mask = (
    (df["Duration"] < lower_bound)
    | (df["Duration"] > upper_bound)
)

print("Duration outliers flagged:", int(outlier_mask.sum()))
print("Outliers have been flagged, not automatically removed.")

Duration/timestamp mismatches greater than 2 minutes: 2658
Duration outliers flagged: 3990
Outliers have been flagged, not automatically removed.


In [14]:
# Create output directory
OUTPUT_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save cleaned dataset
cleaned_path = OUTPUT_DIR / "ChargingRecords_cleaned.csv"

df.to_csv(
    cleaned_path,
    index=False
)

# Save suspicious duration records separately
anomaly_path = (
    OUTPUT_DIR / "duration_anomalies_for_review.csv"
)

duration_anomalies.to_csv(
    anomaly_path,
    index=False
)

# Prepare a cleaning summary
cleaning_report = pd.DataFrame({
    "Metric": [
        "Initial Records",
        "Initial Columns",
        "Initial Missing Cells",
        "Initial Duplicate Rows",
        "Duplicate Rows Removed",
        "Rows Missing Essential Values Removed",
        "Negative Demand Rows Removed",
        "Invalid Sessions Removed",
        "Duration Values Repaired",
        "Zero-Length Rows Removed",
        "Other Unresolved Duration Rows Removed",
        "Final Records",
        "Final Columns",
        "Final Missing Cells",
        "Duration Mismatches Flagged",
        "Duration Outliers Flagged"
    ],
    "Value": [
        initial_rows,
        len(raw_df.columns),
        initial_missing_cells,
        initial_duplicate_rows,
        duplicates_removed,
        missing_rows_removed,
        negative_demand_removed,
        invalid_sessions_removed,
        duration_values_repaired,
        zero_length_rows_removed,
        other_unresolved_rows_removed,
        len(df),
        len(df.columns),
        int(df.isnull().sum().sum()),
        int(mismatch_mask.sum()),
        int(outlier_mask.sum())
    ]
})

report_path = OUTPUT_DIR / "cleaning_report.csv"

cleaning_report.to_csv(
    report_path,
    index=False
)

print("DATA CLEANING COMPLETED")
print("-" * 40)

print("Original dataset rows:", initial_rows)
print("Cleaned dataset rows:", len(df))
print("Cleaned dataset columns:", len(df.columns))

print("\nSaved files:")
print("Cleaned data:", cleaned_path)
print("Duration anomalies:", anomaly_path)
print("Cleaning report:", report_path)

display(cleaning_report)

DATA CLEANING COMPLETED
----------------------------------------
Original dataset rows: 72856
Cleaned dataset rows: 72799
Cleaned dataset columns: 13

Saved files:
Cleaned data: c:\Users\kavee\OneDrive\Desktop\EV Charging Demand Prediction and Usage Pattern Analysis\data\processed\ChargingRecords_cleaned.csv
Duration anomalies: c:\Users\kavee\OneDrive\Desktop\EV Charging Demand Prediction and Usage Pattern Analysis\data\processed\duration_anomalies_for_review.csv
Cleaning report: c:\Users\kavee\OneDrive\Desktop\EV Charging Demand Prediction and Usage Pattern Analysis\data\processed\cleaning_report.csv


,Metric,Value
0,Initial Records,72856
1,Initial Columns,13
2,Initial Missing Cells,0
3,Initial Duplicate Rows,13
4,Duplicate Rows Removed,13
5,Rows Missing Essential Values Removed,0
6,Negative Demand Rows Removed,0
7,Invalid Sessions Removed,17
8,Duration Values Repaired,691
9,Zero-Length Rows Removed,27
